In [1]:
from collections import Counter

In [55]:
file_p = "../text.txt"

with open(file_p, "r") as f:
    txt = f.read()
words= txt.split()
words = words[:10000]

In [56]:
word_counts = Counter(words)
filtered_words = [word for word in words if word_counts[word]>0]
len(filtered_words), len(set(filtered_words))

(10000, 2927)

In [57]:
vocab = list(set(filtered_words))
vocab.sort()
vocab_len= len(vocab)
total_words = len(filtered_words)

In [74]:
encoding = {word: i for i, word in enumerate(vocab)} # Remove un commone words 
encoding["parichay"]

2231

In [75]:
decoding = {i: word for i, word in enumerate(vocab)}
decoding[546]

'Parichay'

In [61]:
wordIds_to_tokens = lambda x:[encoding[filtered_words[i]] for i in x]
wordIds_to_tokens([10,11, 12])

[1082, 1053, 2360]

In [76]:
import numpy as np
context_sz = 2  # 2 words to left, 2 words to right. 

def getData(bs=8):
    trgt = [np.random.randint(0, total_words-2) for i in range(1)]
    context = [[i-2, i-1, i+1, i+2] for i in trgt]

    trgt_tokens = wordIds_to_tokens(trgt)
    context_tokens = wordIds_to_tokens(context[0])
    return (trgt_tokens, context_tokens)

getData()


([86], [722, 258, 155, 1943])

In [63]:
import torch
import torch.nn as nn

class Embeddings(nn.Module):
    def __init__(self, vocab_sz, v_dim):
        super(Embeddings, self).__init__()
        self.embeddings = nn.Embedding(vocab_sz, v_dim)
        self.l1 = nn.Linear(v_dim, 128)
        self.a1 = nn.ReLU()
        self.l2 = nn.Linear(128, vocab_sz)
        self.a2 = nn.LogSoftmax(dim =-1)

        
    def forward(self, embeds):
        out = self.l1(embeds)
        out = self.a1(out)
        out = self.l2(out)
        out = self.a2(out)
        return out

    def get_word_embeddings(self, token):
        if not isinstance(token, torch.Tensor):
            token = torch.tensor(token, dtype=torch.long)
        return self.embeddings(token)
        

    def context_mean(self, ints):
        embeds = torch.stack([self.get_word_embeddings(i) for i in ints])
        return embeds.mean(dim=0) # DIV by 4 as context_sz is 2 to get the mean

In [77]:
model = Embeddings(vocab_len, 100)
loss_fn = nn.NLLLoss()
optim = torch.optim.SGD(model.parameters(), lr=0.001)

In [86]:
for i in range(5):
    for i in range(200):
        total_loss = 0
        optim.zero_grad()
        
        for _ in range(8):
            trgt, context_vect = getData()
            embeds = model.context_mean(context_vect)
            log_preds = model(embeds)
            trgt_tensor = torch.tensor([trgt], dtype=torch.long)  # Shape: (1,)
            loss = loss_fn(log_preds.unsqueeze(0), trgt_tensor[0])
            
            loss.backward()
            total_loss += loss.item()
            
        optim.step()
        print(f"Step: {i}, Loss: {total_loss}")

Step: 0, Loss: 42.40569895505905
Step: 1, Loss: 57.50588274002075
Step: 2, Loss: 34.91542261838913
Step: 3, Loss: 31.164130747318268
Step: 4, Loss: 43.10351845622063
Step: 5, Loss: 40.883944272994995
Step: 6, Loss: 63.49578285217285
Step: 7, Loss: 26.572440207004547
Step: 8, Loss: 51.56140422821045
Step: 9, Loss: 55.91882663965225
Step: 10, Loss: 57.166310369968414
Step: 11, Loss: 48.075109750032425
Step: 12, Loss: 44.643399864435196
Step: 13, Loss: 50.90997850894928
Step: 14, Loss: 44.951767921447754
Step: 15, Loss: 40.021992802619934
Step: 16, Loss: 35.52598196268082
Step: 17, Loss: 53.86061817407608
Step: 18, Loss: 41.91122218966484
Step: 19, Loss: 55.45227724313736
Step: 20, Loss: 34.654899060726166
Step: 21, Loss: 52.129679679870605
Step: 22, Loss: 55.469308853149414
Step: 23, Loss: 37.80920672416687
Step: 24, Loss: 47.40737557411194
Step: 25, Loss: 53.90166223049164
Step: 26, Loss: 34.66059863567352
Step: 27, Loss: 43.75439837574959
Step: 28, Loss: 29.11826664209366
Step: 29, Los

In [25]:
log_preds.shape, trgt_tensor[0].shape

(torch.Size([5196]), torch.Size([1]))

In [26]:
t1 = encoding["parichay"]
t2 = encoding["Pari"]
t1, t2

(3877, 937)

In [37]:
t1_ = model.embeddings.weight[t1]
t2_ = model.embeddings.weight[t2]
(t1_-t2_).sum()

tensor(-25.3652, grad_fn=<SumBackward0>)

In [87]:
import word2vec

ModuleNotFoundError: No module named 'word2vec'